动态规划



In [ ]:
# # drop duplicates
# list(set([1,2,2]))

# # create a matrix of size m x n filled with 1s
# M=[[1]*10 for _ in range(5)]

# # float
# float('inf')

# random
import random
# generate a random integer between 1 and 10
random.randint(0, 10)
# generate a random float between a and b
# random.uniform(a, b)
# generate a random element from a list
# random.choice([1, 2, 3, 4, 5])
# generate a random list
# [random.randint(1, 10) for _ in range(5)]
# generate a random list of floats between a and b
# [random.uniform(a, b) for _ in range(5)]

## 1. 走台阶

In [ ]:
# 走台阶
# N级台阶 每次走S级（S is a list of possible steps），求走法总数

"""
State transition function
S[i] = S[i-s] for s in S if i-s >= 0
"""


def climb_stairs(N, S):
    dp = [0]*(N+1) # initial value 0: means unreachable
    dp[0] = 1 
    for i in range(1, N+1):
        for s in S:
            if i - s >= 0:
                dp[i] += dp[i-s]
    return dp[N]


# Example usage
N = 10
S = list(set([1,2])) # no duplicates
climb_stairs(N, S)



## 2. 连续子数组最大和

In [ ]:
# 连续子数组最大和
# 给定一个整数数组，找到一个具有最大和的连续子数组（子数组最少包含一个元素），返回其最大和

"""
S[i]: means the maximum sum of a subarray ending at index i (must end at i)
S[i] = max(S[i-1] + nums[i], nums[i])
"""

def max_subarray(nums):
    if not nums:
        return 0

    dp = [None] * len(nums)
    dp[0] = nums[0]
    for i in range(1, len(nums)):
        dp[i] = max(dp[i-1] + nums[i], nums[i])
    return max(dp)

# Example usage
nums = [-2,1,-3,4,-1,2,1,-5,4]
max_subarray(nums)

### 环形

In [ ]:
# 如果这个数组是一个cricle (在计算最大子序列和时 环上的元素不能被考虑多次，不能套圈)


"""
假如这个数组是nums
可以认为这个数组从i=0这里剪开了

那么环形数组的最大sum subsequence落在这个list上只有两种情况
# 1. 最大子序列不跨越数组首尾
-> 可以直接用普通的最大子序列和算法求解

# 2. 最大子序列跨越数组首尾
-> 可以通过求整个数组的和减去中间的最小子序列和来得到

"""

def max_subarray_circle(nums):
    if not nums:
        return 0

    # Case 1: Maximum subarray does not cross the array's end
    max_sum = [None] * len(nums)  # means the maximum sum of a subarray ending at index i
    max_sum[0] = nums[0]
    for i in range(1, len(nums)):
        max_sum[i] = max(max_sum[i-1] + nums[i], nums[i])

    min_sum = [None] * len(nums)  # means the minimum sum of a subarray ending at index i
    min_sum[0] = nums[0]
    for i in range(1, len(nums)):
        min_sum[i] = min(min_sum[i-1] + nums[i], nums[i])

    total_sum = sum(nums)

    MAX = max(max_sum)
    MIN = min(min_sum)
    if MAX >= total_sum - MIN:
        return MAX
    else:
        if total_sum == MIN: # the subsequence cannot be empty
            return MAX
        return total_sum - MIN


## 3. 数组最长上升子序列

### 连续

In [ ]:
# 数组最长上升连续子序列

"""
Longest Continuous Increasing Subsequence (LCIS)
Given an integer array nums, return the length of the longest continuous strictly increasing subsequence.

S[i] the length of the longest continuous increasing subsequence ending at index i
S[i] = S[i-1] + 1 if nums[i] > nums[i-1] else 1
"""

def length_of_lcis(nums):
    if not nums:
        return 0
    
    dp = [1] * len(nums)
    for i in range(1, len(nums)):
        if nums[i] > nums[i-1]:
            dp[i] = dp[i-1] + 1
        else:
            dp[i] = 1
    return max(dp)

# Example usage
nums = [10, 9, 2, 5, 3, 7, 101, 18]
print(length_of_lcis(nums))

### 不连续

In [ ]:
# 数组最长上升子序列（可以不连续）

"""
Longest Increasing Subsequence (LIS) problem
Given an integer array nums, return the length of the longest strictly increasing subsequence.

S[i] is the length of the longest increasing subsequence ending at index i
S[i] = max([
    S[j] + 1 if nums[j] < nums[i] else 1
    for j in range(i)
])

动态规划可以解 O(n^2)
但想要拿具体的list可以dfs
"""

def length_of_lis(nums):
    if not nums:
        return 0

    dp = [1] * len(nums)
    for i in range(len(nums)):
        for j in range(i):
            if nums[j] < nums[i]:
                dp[i] = max(dp[i], dp[j] + 1)

    return max(dp)

# Example usage
nums = [10, 9, 2, 5, 3, 7, 101, 18]
print(length_of_lis(nums))

### 连续环形数组

In [9]:
"""
1. 最长上升连续子序列没有cross boundary

2. cross boundary

left_longest[i]: consider the elements [0, i) the length of the longest continuous increasing subsequence starting from 0
right_longest[i]: consider the elements [i, n) the length of the longest continuous increasing subsequence ending at n-1

cross boundary的最长序列:
max(left_longest[i] + right_longest[i] for i in range(1, n - 1))


"""

def length_of_lcis_circular(nums):

    n = len(nums)
    if n<=1:
        return len(nums)

    # non-cross boundary
    dp = [1] * n
    for i in range(1, n):
        if nums[i] > nums[i-1]:
            dp[i] = dp[i-1] + 1
        else:
            dp[i] = 1
    max_length = max(dp)  

    # cross boundary
    if nums[0] > nums[-1]: 

        left_longest = [0] * n
        increasing = True
        cnt = 1
        for i in range(1, n):
            if nums[i] > nums[i - 1] and increasing:
                cnt += 1
            else:
                increasing = False
            left_longest[i] = cnt

        right_longest = [0] * n
        decreasing = True
        cnt = 1
        right_longest[n-1] = 1
        for i in range(n - 2, -1, -1):
            if nums[i] < nums[i + 1] and decreasing:
                cnt += 1
            else:
                decreasing = False
            right_longest[i] = cnt

        for i in range(1, n):
            max_length = max(max_length, left_longest[i] + right_longest[i])
    
    return max_length

length_of_lcis_circular([1,2,4,5,2,0])

5

### 最大回撤

In [ ]:
# 最大回撤
# 一个序列的MDD

"""
Max[i]: Max values before position i [0, i]
Pos[i]: position of the max value before position i

Max[i] = max(Max[i-1], prices[i])
Pos[i] = 
    i if prices[i] > Max[i-1]
    Pos[i-1] if prices[i] < Max[i-1]
    Pos[i-1] if prices[i] == Max[i-1] # keep the the oldest max position
    i if prices[i] == Max[i-1] # keep the the latest max position


MDD[i]: drawdown at position i [0, i] = Max[i] - prices[i]

"""

# only calculate the MDD
# def max_drawdown(prices):
#     if not prices:
#         return 0

#     max_price = [float('-inf') for _ in range(len(prices))]
#     max_price[0] = prices[0]
#     max_dd = [0 for _ in range(len(prices))] # drawdown at position i

#     for i in range(1, len(prices)):
#         max_price[i] = max(max_price[i-1], prices[i])
#         max_dd[i] = max_price[i] - prices[i]

#     return max(max_dd)


# Both MDD and MDD period
def max_drawdown(prices):
    if not prices:
        return 0, (0, 0)

    max_price = prices.copy()
    max_pos = [0 for _ in range(len(prices))] # position of the max value before position i
    max_dd = [0 for _ in range(len(prices))] # drawdown at position i
    MDD = 0
    MDD_PERIOD = (0,0) # both inclusive

    for i in range(1, len(prices)):
        if prices[i] > max_price[i-1]: # >= then record the lastest max position in duplicates
            max_price[i] = prices[i]
            max_pos[i] = i
        else:
            max_price[i] = max_price[i-1]
            max_pos[i] = max_pos[i-1]

        max_dd[i] = max_price[i] - prices[i]
        if max_dd[i] > MDD: # >= then record the lastest MDD and MDD_PERIOD in duplicates
            MDD = max_dd[i]
            MDD_PERIOD = (max_pos[i], i)


    # # clean up version: space O(1) -> O(n)
    # max_price = prices[0]
    # max_pos = 0
    # MDD = 0
    # MDD_PERIOD = (0, 0)

    # for i in range(1, len(prices)):
    #     if prices[i] > max_price:
    #         max_price = prices[i]
    #         max_pos = i

    #     drawdown = max_price - prices[i]

    #     if drawdown > MDD:
    #         MDD = drawdown
    #         MDD_PERIOD = (max_pos, i)

    return MDD, MDD_PERIOD


# Example usage
prices = [100, 120, 90, 150, 80, 200]
max_drawdown(prices)


## 4. 数字和为sum的方法数

In [ ]:
# 4.1 不重复元素的数组中，求和为target的组合方法数 （每个数只能使用一次）
# p.s. 组合的意思是说 (1,2,3) 和 (3,2,1) 算作同一种组合

"""
S[i][j]: 表示在position [0,i-1]的元素中，选取若干元素使其和为 j 的组合方法数

S[i][j] = 
    S[i-1][j] # use 0..i-2 elements to form sum j
    + S[i-1][j-nums[i-1]] if j-nums[i-1] >= 0 else 0 # use 0..i-2 elements to form sum j-nums[i-1]

S[:][0] = 1
S[0][j] = 0 for j > 0

"""

def count_combinations(nums, target):
    if target < 0:
        return 0
    n = len(nums)
    S = [[0] * (target + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        S[i][0] = 1

    for i in range(1, n + 1):
        for j in range(1, target + 1):
            S[i][j] = S[i-1][j] + (S[i-1][j-nums[i-1]] if j-nums[i-1] >= 0 else 0)
    return S[n][target]

# # reduce the space to O(target) with a one-dimensional DP
# def count_combinations(nums, target):
#     # dp[i] represents the number of combinations to form sum 'i' using the elements seen so far
#     dp = [0] * (target + 1)
#     dp[0] = 1
#     for num in nums:
#         for total in range(target, num - 1, -1): # move backward from target to num
#             dp[total] += dp[total - num]
#     return dp[target]

# Example usage
nums = [1, 2, 3, 4] # does not need to be sorted
target = 5
count_combinations(nums, target)


In [ ]:
# 4.2 不重复元素的数组中，求和为target的组合方法数 （每个数可以任意使用）

"""
S[i][j]: 表示在position [0,i-1]的元素中，选取若干元素使其和为 j 的组合方法数

S[i][j] = 
    S[i-1][j] # use 0..i-2 elements ONLY to form sum j
    + S[i][j-nums[i-1]] if j-nums[i-1] >= 0 else 0 # use 0..i-2 elements and AT LEAST one i-1 element to form sum j

S[:][0] = 1
S[0][j] = 0 for j > 0

"""

def count_combinations(nums, target):
    if target < 0:
        return 0
    n = len(nums)
    S = [[0] * (target + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        S[i][0] = 1

    for i in range(1, n + 1):
        for j in range(1, target + 1):
            S[i][j] = S[i-1][j] + (S[i][j-nums[i-1]] if j-nums[i-1] >= 0 else 0)
    return S[n][target]

# Example usage
nums = [1, 2, 3, 4] # does not need to be sorted
target = 5
count_combinations(nums, target)

In [ ]:
# 4.3 在一个数组中，有重复元素，重复几次代表可以用几次，求和为target的组合方法数
# p.s. 组合的意思是说 (1,2,1) 和 (1,1,2) 算作同一种组合

"""
original nums # [1, 1, 3, 2, 3, 4]
counter = Counter(nums)
counts = list(counter.values()) # [2, 2, 1, 1]
values = list(counter.keys()) # [1, 3, 2, 4]

S[i][j]: 表示在values中position [0,i-1]的元素中，选取若干元素使其和为 j 的组合方法数

S[i][j] = 
    S[i-1][j] # use 0..i-2 elements ONLY to form sum j
    + S[i-1][j-values[i-1]] if j-values[i-1] >= 0 else 0
    + S[i-1][j-2*values[i-1]] if j-2*values[i-1] >= 0 else 0
    ...
    + S[i-1][j-counts[i-1]*values[i-1]] if j-counts[i-1]*values[i-1] >= 0 else 0

S[:][0] = 1
S[0][j] = 0 for j > 0    

"""
from collections import Counter
def count_combinations(nums, target):
    if target < 0:
        return 0

    counter = Counter(nums)
    counts = list(counter.values())
    values = list(counter.keys())

    n = len(values)
    S = [[0] * (target + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        S[i][0] = 1

    for i in range(1, n + 1):
        for j in range(1, target + 1):
            S[i][j] = S[i-1][j]
            for k in range(1, counts[i-1] + 1):
                if j - k * values[i-1] >= 0:
                    S[i][j] += S[i-1][j - k * values[i-1]]
    return S[n][target]


# Example usage
nums = [1, 1, 3, 2, 3, 4] # does not need to be sorted
target = 5
count_combinations(nums, target)

## 5. 换零钱

In [ ]:
# 零钱兑换，凑成金额最小数量

# 每个零钱仅用一次
# 每个零钱限次数 很类似，直接repeat in coins列表相应的次数

"""
S[i]: 表示凑成金额i所需的最少硬币数

S[i] = min(
    S[i-coin] + 1 for coin in coins if i >= coin
)

S[0] = 0
"""
def coin_change(coins, target):
    S = [float('inf')] * (target + 1)
    S[0] = 0
    for i in range(1, target + 1):
        for coin in coins:
            if i >= coin:
                S[i] = min(S[i], S[i - coin] + 1)
    return S[target] if S[target] != float('inf') else -1

coins = [1, 2, 5] 
target = 11
print(coin_change(coins, target))

In [ ]:
# 每个零钱不限次数

"""
S[i][j]: 表示只使用前i种硬币 凑成金额j所需的最少硬币数

S[i][j] = min(
    S[i-1][j],  # 不使用第i种硬币
    S[i][j-coin] + 1 for coin in coins if j >= coin  # 使用第i种硬币
)

S[:][0] = 0
S[0][j] = float('inf') if j>0
"""

def coin_change_unlimited(coins, target):
    n = len(coins)
    S = [[float('inf')] * (target + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        S[i][0] = 0
    for i in range(1, n + 1):
        for j in range(1, target + 1):
            S[i][j] = S[i-1][j]
            if j >= coins[i-1]:
                S[i][j] = min(S[i][j], S[i][j-coins[i-1]] + 1)
    return S[n][target] if S[n][target] != float('inf') else -1

coins = [1, 2, 5] 
target = 11
print(coin_change_unlimited(coins, target))

## 6. 背包问题

In [ ]:
# 0/1背包问题，求在容量为c的情况下，物品的最大价值

# 6.1 0/1 knapsack problem: 每个物品只能用一次
# 6.2 每个物品可用多次，但有次数限制 -> 只需要repeat一下list里面相应的元素相应的次数

"""
S[i][j]: 表示前i个物品在容量为j的情况下的最大价值

S[i][j] = max(
    S[i-1][j],  # 不选第i个物品
    S[i-1][j-w[i-1]] + v[i-1] if j>=w[i-1] else 0  # 选第i个物品
)

S[:][0] = 0
S[0][:] = 0

"""

def knapsack_once(w, v, c):
    n = len(w)
    
    m = [[0]*(c+1) for _ in range(n+1)]
    for i in range(1, n+1):
        for j in range(1, c+1):
            if j >= w[i-1]:
                m[i][j] = max(m[i-1][j], m[i-1][j-w[i-1]] + v[i-1])
            else:
                m[i][j] = m[i-1][j]
    return m[n][c]


w=[1,2,5,9]
v=[1,3,8,20]
c=15
print(knapsack_once(w, v, c))


In [ ]:
# 完全背包问题每个物品可用多次（不限次数）
"""
S[i][j]: 表示前i个物品在容量为j的情况下的最大价值

S[i][j] = max(
    S[i-1][j],  # 不选第i个物品
    S[i][j-w[i-1]] + v[i-1] if j>=w[i-1] else 0  # 选第i个物品 可重复选择
)

S[:][0] = 0
S[0][:] = 0

"""

def knapsack_complete(w, v, c):
    n = len(w)
    
    m = [[0]*(c+1) for _ in range(n+1)]
    for i in range(1, n+1):
        for j in range(1, c+1):
            if j >= w[i-1]:
                m[i][j] = max(m[i-1][j], m[i][j-w[i-1]] + v[i-1])
            else:
                m[i][j] = m[i-1][j]
    return m[n][c]

## 7. 接雨水


In [ ]:
"""
Trapping Rain Water (TRW)
Given n non-negative integers representing an elevation map where the width of each bar is 1, compute how much water it can trap after raining.



"""



def trap(height):
    if not height:
        return 0
    
    n = len(height)
    leftMax = [height[0]] + [0] * (n - 1)
    for i in range(1, n):
        leftMax[i] = max(leftMax[i - 1], height[i])

    rightMax = [0] * (n - 1) + [height[n - 1]]
    for i in range(n - 2, -1, -1):
        rightMax[i] = max(rightMax[i + 1], height[i])

    ans = sum(min(leftMax[i], rightMax[i]) - height[i] for i in range(n))
    return ans


## 8. 套信封

类似于LIS

In [ ]:
# 套信封  a的长和宽都小于信封 b，那么信封 a可以放到信封 b里 O(nlogn)
def maxEnvelopes(envelopes) -> int:
    def Env_cmp(x, y): # 比较两个信封大小 宽度递增，相同宽度下高度递减
        # 为什么要这么排，因为这样排完以后就可以只看高度不看宽度了，因为相同宽度下，更大的信封一定排在前面，避免了相同宽度时嵌套的bug
        if x[0] < y[0]:
            return -1
        elif x[0] == y[0]:
            if x[1] >= y[1]:
                return -1
            else:
                return 1
        else:
            return 1
    import functools
    envelopes.sort(key=functools.cmp_to_key(Env_cmp)) # 快排的复杂度差不多在nlogn
    print(envelopes)
    # 排序完之后， 就是一个最长上升子序列问题：这时候我们只看高度就可以了
    # 然后这个问题可以用动态规划做，更好的一种做法是贪心+二分查找 时间复杂度：O(nlogn) 空间O(n)
    dp = [] # dp[i]表示长度为 i+1 的递增子序列的末尾元素的最小值，这个dp肯定是一个递增的数组
    '''
    维护一个单调递增的dp数组
    遍历一次arr数组中所有的数，如果dp数组为空或dp数组末尾值小于arr[i]，直接加在后面。
    否则，二分法找到当前数在dp数组中的位置，替换掉原来位置的数。
    最后，dp的长度就是最终最长递增子序列的长度
    例：7 6 8 9 10 0 1 2 3 4 5 
    '''
    for pair in envelopes:
        h = pair[1]
        if not dp or h > dp[-1]:
            dp.append(h)
        else:
            left, right = 0, len(dp)
            while left < right:
                mid = (left + right) // 2
                if h > dp[mid]:
                    left = mid + 1
                elif h <= dp[mid]:
                    right = mid
            dp[left] = h
    return len(dp)
print(maxEnvelopes([[3,4],[2,3],[4,5],[1,3],[2,2],[3,6],[1,2],[3,2],[2,4]]))
